## Langevin dynamics

### Modelos de difusión para IA generativa: Mixturas de gaussianas

Se dice que la distribución del vector aleatorio $\mathbf{X} \in \mathbb{R}^D$ es una mezcla de $K$ gaussianas si
$$
\mathbf{X} \sim \mathcal{N}\left(\boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k \right) \quad \text{ con probabilidad } p_k > 0, \quad k = 1, \ldots, K.
$$
donde $\boldsymbol{\mu}_k \in \mathbb{R}^D$ es la media y  $\boldsymbol{\Sigma}_k \in \mathbb{R}^D \times \mathbb{R}^D$ la matriz de covarianzas de la Gaussiana $k$ en la mixtura, con $\sum_{k=1}^K p_k = 1$. 




NOTA: Para realizar este ejercico, hay que adaptar los ejemplos 

* Demo: Generating samples for a 1D-Gaussian distribution
* Demo: Generating samples for a 2D-Gaussian distribution

en el cuaderno de Python:
   
    demo_langevin_dynamics.ipynb

#### Ejercicio 1.

1. Proporciona la expresión para la pdf de una mezcla de $K$ gaussianas en $D$ dimensiones. 
2. A partir de la pdf, deriva la expresión de la función de score para una mezcla de $K$ gaussianas en $D$ dimensiones.
$$X_k \sim \mathcal{N}(\mu, \Sigma), \quad \text{pdf}(X_k) = \frac{1}{(2\pi|\Sigma|)^{D/2}} \exp\left[ (x-\mu)^T \Sigma^{-1} (x-\mu) \right]$$

La mixtura de gaussianas con pesos $p_k$ será:

$$\text{pdf}\left(\sum_{k=1}^{K} X_k\right) = \sum_{k=1}^{K} p_k \frac{1}{(2\pi|\Sigma_k|)^{D/2}} \cdot \exp\left[ (x-\mu_k)^T\Sigma_k^{-1} (x-\mu_k) \right]$$

Con

$$\sum_{k=1}^{K} p_k = 1$$

El score será $\frac{\partial}{\partial x} \text{pdf}\left(\sum_{k=1}^{K}p_k X_k\right)$:

$$\frac{\partial}{\partial x} \log\left(p\left(\sum_{k=1}^{K} X_k\right)\right) = \sum_{k=1}^{K} p_k \cdot 2 \cdot (x-\mu_k)\Sigma_k^{-1}$$
   

#### Ejercicio 2.

Adapta el ejemplo "Demo: Generating samples for a 1D-Gaussian distribution" para generar muestras de una mezcla de Gaussianas con los parámetros
   $$
   \begin{array}{lll}
   p_1 = 0.3, & \mu_1 = -2.0, &  \sigma_1 = 1.5, \\
   p_2 = 0.7, & \mu_2 =  5.0, & \sigma_2 = 2.5.
   \end{array}
   $$

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm, multivariate_normal

from matplotlib import animation
from IPython.display import HTML

from langevin_dynamics import (
    animation_pdf_discrete,
    simulate_langevin_dynamics_euler_maruyama,
    countour_plot_force_field,
)


In [2]:
def norm_score_function(x, mu, sigma):
    return -  (x - mu) / sigma**2

def difusion(t):
    return np.sqrt(2.0)

n_simulations = 10000

t_0 = 0.0
T = 20.0

score_function = lambda x, t: 0.3*norm_score_function(x, -2.0, 1.5) + 0.7*norm_score_function(x, 5, 2.5)

rng = np.random.default_rng(seed=1233) 
  
a = 5.0
x_min =  0.3*(-2.0) + 0.7*(5) - a * (0.3*1.5 + 0.7*2.5) 
x_max =  0.3*(5) + 0.7*(5) + a * (0.3*1.5 + 0.7*2.5) 
x_0 = x_min + (x_max - x_min) * rng.random(n_simulations)
 
n_steps = 500

t_end = t_0 + T
t = np.linspace(t_0, t_0 + T, n_steps + 1)
delta_T = t[1] - t[0]

x_t = np.empty((n_simulations, n_steps + 1))

x_t[:, 0] = x_0


n_bins = 100
bin_edges = np.linspace(x_min, x_max, n_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] +  bin_edges[1:])
p_t = np.empty((n_bins, n_steps + 1))

rng = np.random.default_rng()
z = rng.standard_normal((n_simulations, n_steps))

p_t[:, 0], _ = np.histogram(x_t[:, 0], bin_edges, density=True)

 
for n in np.arange(n_steps):
    x_t[:, n + 1] = (
        x_t[:, n]
        + score_function(x_t[:, n], t) * delta_T 
        + np.sqrt(2.0) * np.sqrt(delta_T) * z[:, n]
    )
    p_t[:, n + 1], _ = np.histogram(x_t[:, n + 1], bin_edges, density=True)

times, x_t = simulate_langevin_dynamics_euler_maruyama(
    x_0,
    t_end,
    n_steps,
    score_function, 
    diffusion=difusion,
)


In [ ]:
fig, ax, anim = animation_pdf_discrete(
    bin_centers, 
    p_t, 
    n_steps, 
    model_pdf=lambda x: norm.pdf(x, 0.3*(-2.0) + 0.7*(5), np.sqrt(0.3*1.5**2 + 0.7*2.5**2)
)
plt.close()
HTML(anim.to_jshtml(default_mode='once'))

#### Ejercicio 3
Adapta el ejemplo "Demo: Generating samples for a 2D-Gaussian distribution" para generar muestras de una mezcla de Gaussianas con los parámetros
$$
\begin{array}{lll}
   p_1 = 0.3 & \boldsymbol{\mu}_1 = \left( \begin{array}{c} -4.0 \\ -3.0 \end{array} \right) & \boldsymbol{\Sigma}_1 = \left( \begin{array}{cc} 1.5 & 0.7 \\ 0.7 & 2.0 \end{array} \right) \\
   p_2 = 0.7 & \boldsymbol{\mu}_2 = \left( \begin{array}{c} 5.0 \\ 3.0 \end{array} \right) & \boldsymbol{\Sigma}_2 = \left( \begin{array}{cc} 2.5 & -1.5 \\ -1.5 & 2.0 \end{array} \right) 
\end{array}
$$